In [37]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [19]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test')   \
        .getOrCreate()

In [20]:
!curl -L -o fhvhv_tripdata_2021-01.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0  294M    0  655k    0     0   383k      0  0:13:07  0:00:01  0:13:06  383k
  5  294M    5 17.0M    0     0  6421k      0  0:00:46  0:00:02  0:00:44 6427k
 10  294M   10 30.6M    0     0  6932k      0  0:00:43  0:00:04  0:00:39 6937k
 10  294M   10 30.6M    0     0  6558k      0  0:00:46  0:00:04  0:00:42 6561k
 12  294M   12 37.8M    0     0  6770k      0  0:00:44  0:00:05  0:00:39 7817k
 15  294M   15 44.7M    0     0  6474k      0  0:00:46  0:00:07  0:00:39 8415k
 18  294M   18 54.5M    0     0  7245k      0  0:00:41  0:00:07  0:00:34 7693k
 21  294M   21 62.2M    0     0  7141k      0  0:00:42  0:00:08  0:00:34 7355k
 23  294M   23 70.2M    0     0  7331k      0  0:00

In [21]:
!powershell -command "Get-content fhvhv_tripdata_2021-01.parquet -head 5"

PAR1<HL        
cc``3000fŸñL , ™)Fs    z”SýS/,?øå6 (HV0005HV0003         
LìooW-‘Œ}[âA0AQ4MàJJI®å*G¡”5šz¶	#+'Q�'^MXB­ *X%ïq1·\\\£�=ù¬î†·,…óù?‰AUójš0Yb‘–Þ£=+Šˆ©NY"ØY^{î?Øz™ü £¨ç˜Vqä8t ýMû/1„oÆ}h‹%–G
­é?J_õz£"KÞqZ­TRI­Ž9„ÓøêÎ0€©
‰T¡åS^Å?×C¦”¦Çø‚g–¬û)"RÎqõ


In [22]:
printSchema()

NameError: name 'printSchema' is not defined

In [ ]:
df = spark.read \
    .option("header", "true") \
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [23]:
df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [24]:
df = df.repartition(24)

In [25]:
df = spark.read.parquet('fhvhv_tripdata_2021-01')

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/KMC-PC/notebooks/fhvhv_tripdata_2021-01. SQLSTATE: 42K03

In [26]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [40]:
df \
     .show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [39]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+-------------------+------------+------------+
|pickup_date|   dropoff_datetime|PULocationID|DOLocationID|
+-----------+-------------------+------------+------------+
| 2021-01-16|2021-01-16 15:20:22|         254|          78|
| 2021-01-09|2021-01-09 11:35:06|           7|         232|
| 2021-01-26|2021-01-26 18:38:32|         143|         262|
| 2021-01-02|2021-01-02 14:20:52|         263|          68|
| 2021-01-10|2021-01-10 19:15:27|         171|         131|
| 2021-01-02|2021-01-02 02:49:40|          89|          39|
| 2021-01-24|2021-01-24 14:51:45|         255|          90|
| 2021-01-08|2021-01-08 12:16:28|         246|         265|
| 2021-01-17|2021-01-17 07:07:46|         112|         170|
| 2021-01-11|2021-01-11 14:34:41|         235|          32|
| 2021-01-06|2021-01-06 08:56:33|          89|          91|
| 2021-01-06|2021-01-06 21:15:46|          56|          14|
| 2021-01-01|2021-01-01 07:32:19|          72|          39|
| 2021-01-31|2021-01-31 19:16:20|       